# 总光时延迟 T_GR（加入 HM 项）

本 notebook 将已计算得到的 **PM** 结果表与右侧 **HM** 表按 `gps_time` 对齐，
并将 `T_GR` 改为：

\[
T_{GR} = T_{PM} + T_{HM}
\]

同时输出：
- `T_GR_s = T_PM_s + T_HM_s`
- `rho_total_m = c0 * T_GR_s`

并给出自检项：
- `rho_HM_m = c0 * T_HM_s`
- `rho_sum_m = rho_PM_m + rho_HM_m`
- `rho_diff_m = rho_total_m - rho_sum_m`


In [7]:
import pandas as pd

C0 = 299792458.0

# 文件名：与 notebook 放在同一目录下
TPM_XLSX = "Shapiro_TPM_GNI1B_gamma_dtSR.xlsx"
HM_XLSX  = "MeTp_THM.xlsx"

tpm = pd.read_excel(TPM_XLSX)
hm  = pd.read_excel(HM_XLSX)

print("TPM rows:", len(tpm), "cols:", list(tpm.columns))
print("HM  rows:", len(hm),  "cols:", list(hm.columns))

TPM rows: 86400 cols: ['gps_time', 'T_PM_s', 'rho_PM_m', 'dt_inst_s', 'dt_emit_s', 'dt_recv_corr_s', 'd0_x', 'd0_y', 'd0_z', 'd0_dot_vC_mps', 'd0_dot_vD_mps', 'aC_x_mps2', 'aC_y_mps2', 'aC_z_mps2', 'aC_mag_mps2', 'aD_x_mps2', 'aD_y_mps2', 'aD_z_mps2', 'aD_mag_mps2']
HM  rows: 86400 cols: ['gps_time', 'T_HM', 'dt_inst_s', 'dt_TpMr_s', 'dt_MeTp_s', 'dt_n_s', 'd0_x', 'd0_y', 'd0_z', 'd0_dot_vT_mps', 'd0_dot_vM_mps', 'aC_x_mps2', 'aC_y_mps2', 'aC_z_mps2', 'aC_mag_mps2', 'aD_x_mps2', 'aD_y_mps2', 'aD_z_mps2', 'aD_mag_mps2']


In [8]:
# 兼容不同 HM 列名
hm_col_candidates = ["T_HM", "T_HM_s", "HM"]
hm_col = None
for col in hm_col_candidates:
    if col in hm.columns:
        hm_col = col
        break

if hm_col is None:
    raise KeyError(
        f"在 HM 文件中未找到 HM 列。可用列为: {list(hm.columns)}，"
        f"期望列名之一: {hm_col_candidates}"
    )

if "gps_time" not in tpm.columns:
    raise KeyError(f"TPM 文件缺少 gps_time 列，当前列为: {list(tpm.columns)}")

if "gps_time" not in hm.columns:
    raise KeyError(f"HM 文件缺少 gps_time 列，当前列为: {list(hm.columns)}")

if "T_PM_s" not in tpm.columns:
    raise KeyError(f"TPM 文件缺少 T_PM_s 列，当前列为: {list(tpm.columns)}")

if "rho_PM_m" not in tpm.columns:
    raise KeyError(f"TPM 文件缺少 rho_PM_m 列，当前列为: {list(tpm.columns)}")

hm_use = hm[["gps_time", hm_col]].copy().rename(columns={hm_col: "T_HM_s"})

# 以 gps_time 精确对齐
merged = pd.merge(tpm, hm_use, on="gps_time", how="left")

missing_hm = merged["T_HM_s"].isna().sum()
if missing_hm > 0:
    print(f"Warning: 有 {missing_hm} 行未匹配到 HM，已用 0 填充。")
    merged["T_HM_s"] = merged["T_HM_s"].fillna(0.0)

# 计算 T_GR
merged["T_GR_s"] = merged["T_PM_s"] - merged["T_HM_s"]
merged["delta_t_s"] = merged["T_GR_s"]         # 保留原有输出习惯
merged["rho_HM_m"] = C0 * merged["T_HM_s"]
merged["rho_total_m"] = C0 * merged["T_GR_s"]

# 自检：距离域求和
merged["rho_sum_m"] = merged["rho_PM_m"] + merged["rho_HM_m"]
merged["rho_diff_m"] = merged["rho_total_m"] - merged["rho_sum_m"]

merged.head()

,gps_time,T_PM_s,rho_PM_m,dt_inst_s,dt_emit_s,dt_recv_corr_s,d0_x,d0_y,d0_z,d0_dot_vC_mps,...,aD_y_mps2,aD_z_mps2,aD_mag_mps2,T_HM_s,T_GR_s,delta_t_s,rho_HM_m,rho_total_m,rho_sum_m,rho_diff_m
0,707659200,8.419424e-13,0.000252,0.00065,0.001301,-0.00065,0.534905,0.433562,-0.725190,-7633.447095,...,-4.077941,-5.746020,8.487544,-1.831796e-16,8.421256e-13,8.421256e-13,-5.491587e-08,0.000252,0.000252,1.098317e-07
1,707659201,8.419436e-13,0.000252,0.00065,0.001301,-0.00065,0.535515,0.434087,-0.724425,-7633.446221,...,-4.073931,-5.752991,8.487570,-1.846463e-16,8.421283e-13,8.421283e-13,-5.535558e-08,0.000252,0.000252,1.107112e-07
2,707659202,8.419448e-13,0.000252,0.00065,0.001301,-0.00065,0.536126,0.434612,-0.723659,-7633.445323,...,-4.069917,-5.759955,8.487596,-1.861132e-16,8.421310e-13,8.421310e-13,-5.579534e-08,0.000252,0.000252,1.115907e-07
3,707659203,8.419461e-13,0.000252,0.00065,0.001301,-0.00065,0.536735,0.435136,-0.722892,-7633.444399,...,-4.065897,-5.766912,8.487622,-1.875802e-16,8.421336e-13,8.421336e-13,-5.623514e-08,0.000252,0.000252,1.124703e-07
4,707659204,8.419473e-13,0.000252,0.00065,0.001301,-0.00065,0.537344,0.435659,-0.722124,-7633.443450,...,-4.061872,-5.773862,8.487648,-1.890474e-16,8.421363e-13,8.421363e-13,-5.667500e-08,0.000252,0.000252,1.133500e-07


In [9]:
OUT_XLSX = "LightTime_T_GR_MeTp.xlsx"
merged.to_excel(OUT_XLSX, index=False)

print("Wrote:", OUT_XLSX)
print("rows:", len(merged))
print("rho_diff_m abs max:", merged["rho_diff_m"].abs().max())

show_cols = [c for c in [
    "gps_time", "T_PM_s", "T_HM_s", "T_GR_s",
    "rho_PM_m", "rho_HM_m", "rho_total_m", "rho_sum_m", "rho_diff_m"
] if c in merged.columns]

merged[show_cols].head(10)

Wrote: LightTime_T_GR_MeTp.xlsx
rows: 86400
rho_diff_m abs max: 5.248988906427572e-07


,gps_time,T_PM_s,T_HM_s,T_GR_s,rho_PM_m,rho_HM_m,rho_total_m,rho_sum_m,rho_diff_m
0,707659200,8.419424e-13,-1.831796e-16,8.421256e-13,0.000252,-5.491587e-08,0.000252,0.000252,1.098317e-07
1,707659201,8.419436e-13,-1.846463e-16,8.421283e-13,0.000252,-5.535558e-08,0.000252,0.000252,1.107112e-07
2,707659202,8.419448e-13,-1.861132e-16,8.421310e-13,0.000252,-5.579534e-08,0.000252,0.000252,1.115907e-07
3,707659203,8.419461e-13,-1.875802e-16,8.421336e-13,0.000252,-5.623514e-08,0.000252,0.000252,1.124703e-07
4,707659204,8.419473e-13,-1.890474e-16,8.421363e-13,0.000252,-5.667500e-08,0.000252,0.000252,1.133500e-07
5,707659205,8.419485e-13,-1.905148e-16,8.421390e-13,0.000252,-5.711490e-08,0.000252,0.000252,1.142298e-07
6,707659206,8.419497e-13,-1.919823e-16,8.421416e-13,0.000252,-5.755484e-08,0.000252,0.000252,1.151097e-07
7,707659207,8.419508e-13,-1.934499e-16,8.421443e-13,0.000252,-5.799483e-08,0.000252,0.000252,1.159897e-07
8,707659208,8.419520e-13,-1.949177e-16,8.421469e-13,0.000252,-5.843485e-08,0.000252,0.000252,1.168697e-07
9,707659209,8.419532e-13,-1.963856e-16,8.421496e-13,0.000252,-5.887491e-08,0.000252,0.000252,1.177498e-07
